# AI 시험 연습 문제 - 풀이

Time Series (LSTM/Conv1D/Encoder-Decoder) + Recommender System (NCF/NGCF) 핵심 코드 풀이

## Setup

In [1]:
# [코드]
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 핵심 개념 1. Sliding Window (슬라이딩 윈도우)

**목표**: 1차원 시계열 데이터를 딥러닝 모델이 학습할 수 있는 (입력 시퀀스, 다음 값) 쌍으로 변환.

- `sequence_length = 50` 이면, 과거 50일의 값을 입력 `X`로, 바로 다음 날의 값을 레이블 `y`로 사용.
- 전체 길이가 `N`인 시계열에서 `N - sequence_length` 개의 샘플 생성.
- 인덱스 `i`번째 샘플: `X = data[i:i+50]`, `y = data[i+50]`
- 입력 텐서 shape: `(num_samples, sequence_length, num_features)` → RNN/LSTM 표준 포맷

**왜 중요?**: 시계열은 "다음 값 예측"이 기본 과제이며, sliding window는 모든 시계열 딥러닝 파이프라인의 첫 단계.
### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 파라미터 역할
1. **`data_seq[i:i + sequence_length]`**: 인덱스 `i`부터 `sequence_length`(50)개만큼의 연속 시점을 잘라 입력 윈도우(`X`)를 만듭니다.
2. **`data_seq[i + sequence_length, 0]`**: 윈도우 바로 다음 시점(`i + 50`)의 첫 번째 타깃 변수(열 인덱스 `0`)를 예측 정답(`y`)으로 지정합니다. `len(data_seq) - sequence_length`까지만 반복해야 인덱스 초과 오류(`IndexError`)가 나지 않습니다.

### Problem 1. Sliding Window

In [ ]:
# [코드]
sequence_length = 50  # 과거 50개 시점을 하나의 입력 윈도우로 사용

def convert_data_into_tensors(data_seq):
    features, labels = [], []  # 입력 시퀀스(X)와 다음 시점 타깃(y)을 담을 리스트
    for i in range(len(data_seq) - sequence_length):  # 슬라이딩 윈도우 유효 범위 순회
        features.append(### data_seq[i:i + sequence_length] ### 작성 필요 ###)  # [i, i+50) 구간 50개 시점 추출
        labels.append(### data_seq[i + sequence_length, 0] ### 작성 필요 ###)  # 바로 다음 시점(i+50)의 첫 번째 피처(타깃)
    features, labels = np.array(features), np.array(labels)  # 텐서 변환 전 NumPy 배열로 병합

    features = torch.tensor(features, dtype=torch.float32)  # float32 입력 텐서 (N, 50, F)
    labels = torch.tensor(labels, dtype=torch.float32)  # float32 타깃 텐서 (N,)
    return features, labels

## 핵심 개념 2. LSTM Model (LSTM 모델 구조)

**LSTM (Long Short-Term Memory)**: 장기 의존성 문제를 해결하기 위한 RNN 변형.

**PyTorch `nn.LSTM` 핵심 인자**:
- `input_size`: 각 시점의 feature 개수 (예: 가격만 쓰면 1)
- `hidden_size`: hidden state 차원
- `num_layers`: 쌓을 LSTM 층 수
- `batch_first=True`: 입력을 `(batch, seq_len, input_size)` 형태로 기대

**forward 출력**: `out, (h_n, c_n) = self.lstm(x)`
- `out`: 모든 time step의 hidden state `(batch, seq_len, hidden_size)`
- `h_n, c_n`: 마지막 hidden/cell state
- 예측에는 `out`의 마지막 time step이나 전체를 Linear에 통과시킴.
### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 파라미터 역할
1. **`nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)`**:
   - **`batch_first=True`**: 기본값(`False`)은 `(seq_len, batch, features)`이지만, PyTorch DataLoader 표준 `(batch, seq_len, features)`와 맞추기 위해 반드시 `True`를 전달합니다.
2. **`nn.Linear(hidden_size, 1)`**: LSTM이 출력한 `hidden_size` 차원의 은닉 벡터를 최종 회귀 예측 값(1차원)으로 투영합니다.
3. **`self.lstm(x)` & `self.linear(out)`**: `self.lstm(x)`는 `(모든 시점 출력, (h_n, c_n))` 튜플을 반환하므로 `out, _`로 언패킹한 뒤 선형층에 전달합니다.

### Problem 2. LSTM Model

In [ ]:
# [코드]
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(LSTMModel, self).__init__()
        self.lstm = ### nn.LSTM(input_size, hidden_size, num_layers, batch_first=True) ### 작성 필요 ###  # batch_first=True: (B, T, F) 순서 입력
        self.linear = ### nn.Linear(hidden_size, 1) ### 작성 필요 ###  # LSTM 은닉 벡터를 단일 스칼라 예측값으로 사상

    def forward(self, x):
        out, _ = ### self.lstm(x) ### 작성 필요 ###  # out: 모든 시점 은닉 출력 (B, T, hidden_size), _: (h_n, c_n)
        return ### self.linear(out) ### 작성 필요 ###  # 각 시점 은닉 상태를 출력 차원 1로 선형 투영

## 핵심 개념 3. Training Loop (훈련 루프)

**PyTorch 훈련 5단계 (순서 고정)**:
1. `pred = model(batch_x)` — 순전파
2. `loss = loss_fn(pred, batch_y)` — 손실 계산
3. `optimizer.zero_grad()` — 이전 gradient 초기화 (누적 방지)
4. `loss.backward()` — 역전파로 gradient 계산
5. `optimizer.step()` — parameter 업데이트

**시계열 예측 tip**: LSTM 출력이 `(batch, seq_len, 1)`일 때, 마지막 time step만 사용하려면 `model(x)[:, -1, 0]`.

**`zero_grad` 순서 주의**: `backward()` 앞에 두어야 함. PyTorch는 gradient를 누적하므로 초기화가 필수.
### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 파라미터 역할
1. **`model(batch_x)[:, -1, 0]`**: 시퀀스 모델 출력 `(B, T, 1)` 중 전체 시퀀스를 종합한 **마지막 타임스텝(`-1`)**의 스칼라 예측(`0`)만 슬라이싱해야 `batch_y` `(B,)`와 형상이 일치합니다.
2. **`loss_fn(pred, batch_y)`**: 예측 텐서와 타깃 텐서를 비교해 스칼라 손실값을 계산합니다.
3. **`zero_grad()` → `backward()` → `step()`**: PyTorch는 기울기를 덮어쓰지 않고 누적(`+=`)하므로 매 배치마다 `zero_grad()` 초기화가 필수입니다.

### Problem 3. Training Loop

In [ ]:
# [코드]
def train_one_epoch(model, train_loader, loss_fn, optimizer):
    model.train()  # 드롭아웃/배치정규화 등을 학습 모드로 전환
    total_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)  # GPU/CPU 디바이스 일치
        pred = ### model(batch_x)[:, -1, 0] ### 작성 필요 ###  # 모델 출력 (B, T, 1)에서 마지막 시점(-1)의 스칼라(0) 추출
        loss = ### loss_fn(pred, batch_y) ### 작성 필요 ###  # 예측값과 정답 간 손실(예: MSE) 계산

        ### optimizer.zero_grad() ### 작성 필요 ###  # 이전 배치의 누적 기울기 초기화
        ### loss.backward() ### 작성 필요 ###  # 역전파로 각 파라미터의 gradient 계산
        ### optimizer.step() ### 작성 필요 ###  # 옵티마이저를 통해 가중치 1스텝 갱신

        total_loss += loss.item()  # 스칼라 손실값 누적
    return total_loss / len(train_loader)  # 배치 수로 나눈 에폭 평균 손실 반환

## 핵심 개념 4. Conv1D Model (1D 합성곱)

**`nn.Conv1d` 입력 형태**: `(batch, in_channels, seq_len)` — 채널이 두 번째 차원.

**시계열 데이터는 보통** `(batch, seq_len, features)` 형태이므로, **Conv1D 앞뒤로 `transpose(1, 2)` 필요**:
1. Conv 입력 전: `(batch, seq_len, features)` → `(batch, features, seq_len)` 로 transpose
2. Conv 실행
3. FC/다음 layer 앞: `(batch, out_channels, new_seq_len)` → `(batch, new_seq_len, out_channels)` 로 다시 transpose

**왜?**: `features`가 Conv의 채널에 해당하고, 커널은 시간 축(seq_len)을 따라 sliding 하기 때문.
### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 파라미터 역할
1. **`x.transpose(1, 2)` (전치 2회)**:
   - PyTorch `nn.Conv1d`는 입력 채널(`in_channels`)이 두 번째 축에 오는 `(batch, channels, seq_len)` 형상을 요구합니다.
   - 따라서 `(batch, seq_len, input_size)` 입력의 `dim=1`과 `dim=2`를 바꾼 뒤 Conv1D를 통과시키고, 후속 `nn.Linear`가 마지막 축을 특징 차원으로 처리하도록 다시 `transpose(1, 2)`로 되돌립니다.

### Problem 4. Conv1D Model

In [ ]:
# [코드]
class Conv1DModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(Conv1DModel, self).__init__()
        self.conv1d = nn.Conv1d(in_channels=input_size,  # 입력 특징 수 (채널 축)
                                out_channels=hidden_size,  # 출력 필터(특징) 개수
                                kernel_size=2, stride=1)  # 인접 2시점을 1칸씩 훑는 1D 합성곱
        self.fc = nn.Linear(hidden_size, 1)  # 마지막 은닉 차원을 예측값 1로 변환

    def forward(self, x):
        x = ### x.transpose(1, 2) ### 작성 필요 ###  # (B, T, F) -> (B, F, T)로 변환 (nn.Conv1d 입력 규격 맞춤)
        x = self.conv1d(x)  # 시간 축(T)을 따라 1D 합성곱 연산 수행
        x = ### x.transpose(1, 2) ### 작성 필요 ###  # (B, hidden, T') -> (B, T', hidden)으로 복원 (nn.Linear 입력 규격)
        return self.fc(x)

## 핵심 개념 5. Encoder-Decoder Architecture (Seq2Seq)

**구조**: 입력 시퀀스를 인코더가 하나의 hidden state `h`로 압축 → 디코더가 `h`로부터 타깃 시퀀스를 하나씩 생성.

**Forward 흐름**:
1. `h = self.encoder(source)` — 입력 전체를 읽고 context vector 반환
2. 초기 decoder 입력 = source의 **마지막 time step** `source[:, -1, :].unsqueeze(1)` (shape `(batch, 1, input_size)`)
3. `target_len` 만큼 loop:
   - `out, h = self.decoder(input, h)` — 한 step 생성
   - 생성된 `out`을 다음 step의 입력으로 재사용 (**autoregressive**)
4. `torch.stack(predictions, dim=1)` — time step list를 `(batch, target_len, dim)` 텐서로 합침.

**Autoregressive 생성의 핵심**: 이전 스텝 출력을 현재 스텝 입력으로 feedback → 다중 step 예측 가능.
### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 파라미터 역할
1. **`self.encoder(source)`**: 입력 시퀀스 전체의 정보를 담은 마지막 은닉 상태 `h`를 디코더의 초기 은닉 상태로 넘깁니다.
2. **`source[:, -1, :].unsqueeze(1)`**: 디코더의 첫 스텝 입력 시퀀스로 `source`의 마지막 시점만 뽑되, `batch_first=True` RNN 규격인 3차원 `(B, 1, F)`를 유지하기 위해 `unsqueeze(1)`을 적용합니다.
3. **`input = out` & `torch.stack(predictions, dim=1)`**: 자기회귀(Autoregressive) 생성을 위해 방금 예측한 `out`을 다음 입력으로 재투입하고, `dim=1`(시간 축)으로 쌓아 `(B, target_len, F)` 출력을 만듭니다.

### Problem 5. Encoder-Decoder Forward

In [ ]:
# [코드]
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(EncoderRNN, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)  # 입력 시퀀스 압축 인코더

    def forward(self, x):
        _, h = self.rnn(x)  # 최종 시점의 은닉 상태 h (num_layers, B, hidden) 반환
        return h

class DecoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(DecoderRNN, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, input_size)  # 은닉 상태를 다음 시점 입력 피처 크기로 복원

    def forward(self, x, h):
        out, h = self.rnn(x, h)  # 이전 은닉 상태 h를 이어받아 한 스텝 예측
        out = self.fc(out)
        return out, h

class RNNRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(RNNRNN, self).__init__()
        self.encoder = EncoderRNN(input_size, hidden_size, num_layers)
        self.decoder = DecoderRNN(input_size, hidden_size, num_layers)

    def forward(self, source, target_len):
        h = ### self.encoder(source) ### 작성 필요 ###  # 입력 시퀀스를 요약한 문맥 은닉 상태 h 획득
        predictions = []
        input = ### source[:, -1, :].unsqueeze(1) ### 작성 필요 ###  # 디코더 첫 입력: source 마지막 시점 (B, 1, F)
        for t in range(target_len):  # 미래 target_len 스텝만큼 자기회귀 생성
            out, h = self.decoder(input, h)  # 한 시점 예측 및 은닉 상태 갱신
            predictions.append(out.squeeze(1))  # (B, 1, F) -> (B, F)로 축소해 저장
            input = ### out ### 작성 필요 ###  # 방금 예측한 값을 다음 스텝의 입력으로 피드백
        outputs = ### torch.stack(predictions, dim=1) ### 작성 필요 ###  # (B, target_len, F) 텐서로 결합
        return outputs

## 핵심 개념 6. Neural Collaborative Filtering (NCF)

**아이디어**: 유저/아이템을 잠재 벡터(embedding)로 학습하고, 두 벡터를 MLP에 통과시켜 rating 예측.

**구조**:
- `user_embedding`: `nn.Embedding(n_users, emb_dim=32)` — 유저 ID → 32차원 벡터
- `movie_embedding`: `nn.Embedding(n_movies, 32)`
- Forward에서 두 임베딩을 **`dim=1` 기준으로 concat** → 64차원
- FC → ReLU → FC(→ 1차원) 로 최종 평점 예측

**`nn.Embedding(num_embeddings, embedding_dim)`**: lookup table. 입력은 정수 인덱스 (유저/아이템 ID), 출력은 해당 row의 학습 가능한 벡터.

**dim=1 concat 이유**: `(batch, 32)` + `(batch, 32)` → `(batch, 64)`. dim=0은 배치 축이므로 써선 안 됨.
### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 파라미터 역할
1. **`nn.Embedding(n_users, 32)` & `nn.Embedding(n_movies, 32)`**: 이산 정수 ID(유저/영화)를 밀집된 32차원 연속 잠재 벡터로 변환하는 룩업 테이블입니다.
2. **`torch.cat([user_embedding, movie_embedding], dim=1)`**: 두 `(B, 32)` 임베딩을 특징 축(`dim=1`) 방향으로 이어 붙여 후속 `nn.Linear(64, 32)`의 입력 차원 `64`를 맞춥니다 (`dim=0`이면 배치 방향이 늘어나 에러 발생).

### Problem 6. NCF Model

In [ ]:
# [코드]
class Neural_Collaborative_Filtering(nn.Module):
    def __init__(self, n_users, n_movies):
        super().__init__()
        self.user_embedding = ### nn.Embedding(n_users, 32) ### 작성 필요 ###  # 유저 ID -> 32차원 잠재 벡터 룩업 테이블
        self.movie_embedding = ### nn.Embedding(n_movies, 32) ### 작성 필요 ###  # 아이템 ID -> 32차원 잠재 벡터 룩업 테이블

        self.fc1 = nn.Linear(64, 32)  # 유저(32) + 아이템(32) 결합 벡터(64)를 32차원으로 압축
        self.relu = nn.ReLU()  # 비선형 활성화
        self.fc2 = nn.Linear(32, 1)  # 최종 평점/선호도 예측 스칼라 출력

    def forward(self, users, movies, ratings=None):
        user_embedding = self.user_embedding(users)  # (B, 32)
        movie_embedding = self.movie_embedding(movies)  # (B, 32)
        input_embedding = ### torch.cat([user_embedding, movie_embedding], dim=1) ### 작성 필요 ###  # 특징 축(dim=1)으로 결합 -> (B, 64)

        hidden_feature = self.fc1(input_embedding)
        hidden_feature = self.relu(hidden_feature)
        output = self.fc2(hidden_feature)
        return output

## 핵심 개념 7. Precision@k / Recall@k (추천 시스템 평가)

**정의** (threshold 이상을 "관련 있음(relevant)"으로 간주):
- **Precision@k** = (top-k 중 실제 relevant 개수) / k — 추천한 것 중 얼마나 맞았나
- **Recall@k** = (top-k 중 실제 relevant 개수) / (전체 relevant 개수) — 실제 relevant 중 얼마나 잡아냈나

**계산 순서**:
1. 유저별 (예측 rating, 실제 rating) 리스트를 **예측 rating 내림차순**으로 정렬
2. 전체에서 `true_r >= threshold` 개수 = `n_real_relevant`
3. 상위 k개 중 `true_r >= threshold` 개수 = `n_real_relevant_in_top_k`
4. Precision = `hits / k`, Recall = `hits / n_real_relevant` (분모 0이면 0 처리)

**주의**: `threshold`(예: 3.5) 이상이어야 "긍정 샘플". 아니면 이진 분류 문제가 성립 안 함.
### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 파라미터 역할
1. **`n_real_relevant`**: 임계값(`threshold=3.5`) 이상인 실제 관련 아이템 총 개수(Recall의 분모).
2. **`user_ratings[:k]`**: 예측 점수 기준 내림차순 정렬된 리스트에서 상위 `k`개 추천 항목만 슬라이싱합니다.
3. **`n_real_relevant_in_top_k / k` vs `/ n_real_relevant`**: Precision@k는 분모가 **추천한 개수 `k`**, Recall@k는 분모가 **실제 정답 총수 `n_real_relevant`**라는 점이 핵심 구분 포인트입니다.

### Problem 7. Precision@k & Recall@k

In [ ]:
# [코드]
def compute_precision_recall(user_est_true, k=10, threshold=3.5):
    precisions = dict()
    recalls = dict()
    for user_id, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)  # 모델 예측 점수(est=x[0]) 기준 내림차순 정렬

        n_real_relevant = ### sum((true_r >= threshold) for (_, true_r) in user_ratings) ### 작성 필요 ###  # 전체 중 실제 관련 있는 아이템 총수
        recommended_k = ### user_ratings[:k] ### 작성 필요 ###  # 예측 점수가 가장 높은 상위 k개 추천 슬라이싱
        n_real_relevant_in_top_k = ### sum((true_r >= threshold) for (est, true_r) in recommended_k) ### 작성 필요 ###  # 상위 k개 내 실제 정답 수

        precisions[user_id] = ### n_real_relevant_in_top_k / k ### 작성 필요 ###  # Precision@k: 추천 k개 중 적중 비율
        if n_real_relevant:
            recalls[user_id] = ### n_real_relevant_in_top_k / n_real_relevant ### 작성 필요 ###  # Recall@k: 전체 정답 중 추천된 비율
        else:
            recalls[user_id] = 0  # 실제 관련 아이템이 0개인 유저는 0 처리
    return precisions, recalls

## 핵심 개념 8. NGCF Normalization (그래프 정규화)

**NGCF (Neural Graph Collaborative Filtering)**: 유저-아이템 interaction을 **그래프(bipartite)**로 보고, GNN으로 embedding 전파.

**정규화 항**: 각 엣지 `(u, i)`의 메시지는 두 노드 degree의 기하평균의 역수로 가중.

$$ \text{norm}_{u,i} = \frac{1}{\sqrt{|\mathcal{N}_u|} \cdot \sqrt{|\mathcal{N}_i|}} = \frac{1}{\sqrt{\text{deg}(u) \cdot \text{deg}(i)}} $$

**왜?**: degree가 큰 인기 노드가 지나치게 영향을 주는 것을 방지(symmetric normalization, GCN에서 유래).

- `deg[src]`: 엣지별 src 노드의 degree를 pull
- `deg[dst]`: 엣지별 dst 노드의 degree를 pull
- `torch.sqrt(deg[src] * deg[dst])` 후 역수
### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 파라미터 역할
1. **`1.0 / torch.sqrt(deg[src] * deg[dst])`**:
   - 그래프 합성곱 시 연결 수(차수 `deg`)가 많은 인기 노드로 메시지가 과도하게 증폭되는 것을 억제하기 위해, 출발 노드와 도착 노드 차수의 기하평균 제곱근으로 나누어 대칭 정규화(Symmetric Normalization)를 수행합니다.

### Problem 8. NGCF Layer: Normalization

In [ ]:
# [코드]
def compute_norm(deg, src, dst):
    norm = ### 1.0 / torch.sqrt(deg[src] * deg[dst]) ### 작성 필요 ###  # 대칭 정규화 1/sqrt(deg(u)*deg(i)): 인기 노드의 영향력 과도 팽창 방지
    return norm

## 핵심 개념 9. NGCF Edge Message (엣지 메시지)

**NGCF의 핵심 수식** (src 노드 u 기준으로 이웃 i로부터 받는 메시지):

$$ m_{u \leftarrow i} = W_1 e_i + W_2 (e_i \odot e_u) $$

- `W1 * dst_feat`: 기본 feature 전달 (GCN 유사)
- `W2 * (dst_feat * src_feat)`: **element-wise 곱**으로 affinity(상호작용) 정보 인코딩 — NGCF의 핵심 기여점

이후 `norm` 곱해 degree 정규화 적용.

**양방향 그래프**이므로 dst 노드 i가 src u에게 받는 메시지도 `W1(src_feat) + W2(src_feat * dst_feat)` 로 대칭 계산.

**vs. GCN**: GCN은 `W * e_i`만 전파. NGCF는 `e_i * e_u` 항 추가로 유저-아이템 "친화도"를 더 잘 포착.
### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 파라미터 역할
1. **`self.W1(dst_feat) + self.W2(dst_feat * src_feat)`**:
   - **첫 번째 항 `W1(dst_feat)`**: 이웃 노드 특징 자체의 선형 변환 (일반 GCN 방식).
   - **두 번째 항 `W2(dst_feat * src_feat)`**: 유저-아이템 두 노드의 **원소별 곱(element-wise product)**을 통해 유사도/협업 필터링 상호작용 신호를 명시적으로 인코딩하는 NGCF의 핵심 항입니다.

### Problem 9. NGCF Layer: Edge Messages

In [ ]:
# [코드]
class NGCFLayer(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.1):
        super().__init__()
        self.W1 = nn.Linear(input_dim, output_dim)  # 이웃 자체 정보 선형 변환 가중치
        self.W2 = nn.Linear(input_dim, output_dim)  # 이웃-자기 노드 상호작용(요소별 곱) 변환 가중치
        self.leaky_relu = nn.LeakyReLU(0.2)

    def forward(self, edge_index, node_features, user_num, item_num):
        src, dst = edge_index  # 엣지 출발(유저)과 도착(아이템) 인덱스

        deg = torch.zeros(node_features.size(0), device=node_features.device)  # 노드별 차수 저장 텐서
        deg.index_add_(0, src, torch.ones_like(src, dtype=torch.float))  # 출발 노드 연결 수 누적
        deg.index_add_(0, dst, torch.ones_like(dst, dtype=torch.float))  # 도착 노드 연결 수 누적

        norm = 1.0 / torch.sqrt(deg[src] * deg[dst])  # 대칭 라플라시안 정규화 계수

        src_feat = node_features[src]  # 소스 노드 특징 (E, D)
        dst_feat = node_features[dst]  # 타깃 노드 특징 (E, D)

        edge_messages_for_src = ### self.W1(dst_feat) + self.W2(dst_feat * src_feat) ### 작성 필요 ###  # 이웃 정보 + 협업 신호(상호작용) 메시지
        edge_messages_for_src *= norm.unsqueeze(1)  # (E, 1)로 브로드캐스팅해 정규화 계수 적용

        edge_messages_for_dst = ### self.W1(src_feat) + self.W2(src_feat * dst_feat) ### 작성 필요 ###  # 반대 방향(소스->타깃) 메시지 계산
        edge_messages_for_dst *= norm.unsqueeze(1)

        aggregated_messages = torch.zeros_like(node_features)
        aggregated_messages.index_add_(0, src, edge_messages_for_src)  # 소스 노드로 이웃 메시지 집계
        aggregated_messages[:user_num] += self.W1(node_features[:user_num])  # 자기 자신(self-loop) 변환 항 추가
        aggregated_messages.index_add_(0, dst, edge_messages_for_dst)  # 타깃 노드로 이웃 메시지 집계
        aggregated_messages[user_num:] += self.W1(node_features[user_num:])

        return self.leaky_relu(aggregated_messages)

## 핵심 개념 10. NGCF Final Feature (계층별 concatenation)

**NGCF의 최종 representation**: layer 0 ~ layer L의 모든 embedding을 **feature 축(dim=-1)으로 concat**.

$$ e^{final} = e^{(0)} \parallel e^{(1)} \parallel \cdots \parallel e^{(L)} $$

- `layer_outputs`는 `[초기 embedding, layer1 out, layer2 out, ...]` 리스트
- `torch.concat(..., dim=-1)` 로 **column-wise** 연결 → 최종 차원 = sum of layer dims
- 상위 `num_users` row → 유저 feature, 하위 → 아이템 feature

**왜 concat?**: 서로 다른 hop의 이웃 정보(local vs. distant)를 모두 보존. 평균이나 sum은 정보 손실 가능.
### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 파라미터 역할
1. **`torch.concat(layer_outputs, dim=-1)`**:
   - 0차 임베딩(자체 ID)부터 1홉, 2홉, ..., L홉 이웃 정보를 집계한 모든 레이어 출력을 특징 차원(`dim=-1`)으로 이어 붙입니다. (단순 합산이 아닌 `concat`을 써서 각 홉 거리의 고유 표현을 소실 없이 모두 보존)

### Problem 10. NGCF: Final Features

In [ ]:
# [코드]
def build_final_features(layer_outputs, num_users):
    final_features = ### torch.concat(layer_outputs, dim=-1) ### 작성 필요 ###  # 0~L층 임베딩을 특징 차원(dim=-1)으로 결합해 다중 홉 정보 보존
    user_features = final_features[:num_users]  # 앞쪽 num_users 개는 유저 최종 임베딩
    item_features = final_features[num_users:]  # 뒷쪽 나머지는 아이템 최종 임베딩
    return user_features, item_features

## 핵심 개념 11. BPR Loss (Bayesian Personalized Ranking)

**BPR 아이디어**: 명시적 rating이 아닌 **implicit feedback** 상황에서, 유저가 상호작용한 아이템(positive)의 점수가 상호작용하지 않은 아이템(negative)보다 높아야 한다는 pairwise 랭킹 손실.

**수식**:
$$ \mathcal{L}_{BPR} = - \frac{1}{N} \sum_{(u, i^+, i^-)} \log \sigma(s_{u, i^+} - s_{u, i^-}) $$

- `pos_scores = sum(user_emb * pos_item_emb, dim=1)` — 유저-positive 아이템 내적
- `neg_scores = sum(user_emb * neg_item_emb, dim=1)` — 유저-negative 아이템 내적
- `-mean(logsigmoid(pos - neg))`: pos가 neg보다 클수록 (차이가 클수록) loss 감소
- `reg_loss`: L2 정규화로 임베딩 크기 폭주 방지

**핵심 포인트**: `dim=1`은 feature 차원(배치별 내적), `F.logsigmoid`는 수치 안정적인 `log(sigmoid(x))`.
### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 파라미터 역할
1. **`torch.sum(user_emb * pos_item_emb, dim=1)`**: 배치 내 각 유저 벡터와 아이템 벡터의 내적(dot product) 점수를 특징 축(`dim=1`) 합으로 계산합니다.
2. **`-torch.mean(F.logsigmoid(pos_scores - neg_scores))`**: 양성 점수(`pos`)가 음성 점수(`neg`)보다 커질수록 `sigmoid`가 1에 가까워지고 `-logsigmoid` 손실이 0으로 수렴하는 쌍별 순위(Pairwise Ranking) 목적식입니다.

### Problem 11. BPR Loss

In [ ]:
# [코드]
def bpr_loss(user_emb, pos_item_emb, neg_item_emb, reg_weight=1e-4):
    pos_scores = ### torch.sum(user_emb * pos_item_emb, dim=1) ### 작성 필요 ###  # 유저-선호 아이템 내적(유사도 점수)
    neg_scores = ### torch.sum(user_emb * neg_item_emb, dim=1) ### 작성 필요 ###  # 유저-비선호 아이템 내적(유사도 점수)
    loss = ### -torch.mean(F.logsigmoid(pos_scores - neg_scores)) ### 작성 필요 ###  # 선호 아이템 점수가 더 높도록 -ln(sigmoid(pos - neg)) 최소화
    reg_loss = reg_weight * (user_emb.norm(2).pow(2) + pos_item_emb.norm(2).pow(2) + neg_item_emb.norm(2).pow(2)) / user_emb.size(0)  # L2 정규화로 과적합 방지
    return loss + reg_loss

## 핵심 개념 12. Scikit-learn 데이터 분할 및 누수 방지 정규화

### 핵심 원리: 데이터 누수(Data Leakage) 방지 전처리
- **정의:** 시계열이나 정형 데이터 전처리 시 평가(Test) 데이터의 통계치(평균, 표준편차 등)가 학습 단계 스케일러에 반영되면 모델 성능이 과대평가되는 Data Leakage가 발생합니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`train_test_split(X, y, test_size=test_size, shuffle=False)`** *(시계열 분할)*:
   - 시계열 데이터는 시간 순서가 중요하므로 무작위 셔플을 끄고(`shuffle=False`) 과거를 Train, 미래를 Test로 나눕니다.
2. **`scaler.fit_transform(X_train)`** *(학습 데이터 기준 통계 학습)*:
   - 학습 데이터에 대해서만 `fit`(평균·표준편차 계산)과 `transform`(스케일링)을 동시에 수행합니다.
3. **`scaler.transform(X_test)`** *(평가 데이터 변환)*:
   - 평가 데이터에는 학습 데이터에서 구한 기준 통계치만 적용해야 하므로 반드시 `fit` 없이 **`transform()`만** 호출해야 합니다.

In [ ]:
# [코드]
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def prepare_scaled_data(X, y, test_size=0.2):
    X_train, X_test, y_train, y_test = ### train_test_split(X, y, test_size=test_size, shuffle=False) ### 작성 필요 ###  # shuffle=False: 시계열 시간 순서 유지 (미래 누수 방지)
    scaler = StandardScaler()  # 평균 0, 분산 1 표준화 스케일러
    X_train_scaled = ### scaler.fit_transform(X_train) ### 작성 필요 ###  # 학습 데이터만으로 평균/표준편차 통계 fit 및 변환
    X_test_scaled = ### scaler.transform(X_test) ### 작성 필요 ###  # 평가 데이터에는 학습 통계만 사용해 변환 (Data Leakage 차단)
    return X_train_scaled, X_test_scaled, y_train, y_test

## 핵심 개념 13. Scikit-learn 회귀 평가 지표 계산 (RMSE / MAE / R2)

### 핵심 원리: 회귀 및 시계열 예측 평가 메트릭
- **정의:** 연속형 예측 모델은 목적에 따라 큰 오차에 민감한 **RMSE**, 이상치에 강건하고 직관적인 **MAE**, 분산 설명력을 나타내는 **$R^2$**를 종합적으로 해석합니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`np.sqrt(mean_squared_error(y_true, y_pred))`** *(RMSE)*:
   - MSE는 오차 제곱의 평균이므로 실제 타깃 단위로 복원하기 위해 제곱근(`np.sqrt`)을 취해 RMSE를 산출합니다.
2. **`mean_absolute_error(y_true, y_pred)`** *(MAE)*:
   - 예측 오차 절댓값의 평균으로, 이상치에 덜 민감하며 직관적인 오차 크기를 보여줍니다.
3. **`r2_score(y_true, y_pred)`** *($R^2$ 결정 계수)*:
   - 1에 가까울수록 타깃 데이터의 분산을 모델이 완벽히 설명함을 의미합니다. (항상 `y_true`가 첫 번째 인자, `y_pred`가 두 번째 인자임에 주의)

In [ ]:
# [코드]
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate_regression(y_true, y_pred):
    rmse = ### np.sqrt(mean_squared_error(y_true, y_pred)) ### 작성 필요 ###  # MSE에 제곱근을 취해 실제 타깃 단위의 오차(RMSE) 산출
    mae = ### mean_absolute_error(y_true, y_pred) ### 작성 필요 ###  # 이상치에 강건한 평균 절대 오차(MAE)
    r2 = ### r2_score(y_true, y_pred) ### 작성 필요 ###  # 결정계수 R^2: 1에 가까울수록 타깃 분산을 완벽히 설명
    return {"rmse": rmse, "mae": mae, "r2": r2}

## 핵심 개념 14. 추천·분류 모델 평가 지표 (ROC-AUC & NDCG@k)

### 핵심 원리: 확률 판별력(ROC-AUC)과 순위 가중 평가(DCG/NDCG)
- **정의:** 추천 시스템에서는 모델이 맞춘 개수뿐만 아니라 **정답 아이템을 얼마나 최상위 순위에 배치했는지**가 중요하며, 이진 선호 예측의 전반적 분류 성능은 ROC 곡선 아래 면적(AUC)으로 평가합니다.

### 🎯 빈칸(`### 작성 필요 ###`)별 출제 포인트 & 원리
1. **`roc_auc_score(y_true, y_scores)`** *(ROC-AUC)*:
   - 임계값(threshold) 변화에 무관하게 모델의 양성/음성 순위 판별 능력을 `0.5 ~ 1.0` 스칼라로 산출합니다. (예측 확률/로짓 점수인 `y_scores` 전달)
2. **`rel / math.log2(i + 2)`** *(DCG 위치 할인)*:
   - 인덱스 `i = 0` (1위)일 때 분모가 $\log_2(0 + 2) = \log_2(2) = 1$이 되어 할인 없이 관련도 점수 `rel`이 온전히 반영되고, 순위가 내려갈수록 로그 반비례로 가중치가 감소합니다.

In [ ]:
# [코드]
import math
from sklearn.metrics import roc_auc_score

def evaluate_ranking_and_auc(y_true, y_scores, ranked_relevances, k):
    auc = ### roc_auc_score(y_true, y_scores) ### 작성 필요 ###  # 예측 점수(y_scores) 기반 전체 이진 분류 판별력(ROC-AUC)
    dcg = 0.0
    for i, rel in enumerate(ranked_relevances[:k]):  # 상위 k개 추천 순위 순회
        dcg += ### rel / math.log2(i + 2) ### 작성 필요 ###  # 순위가 내려갈수록 로그(i+2) 반비례 할인 적용 (i=0일 때 log2(2)=1)
    return auc, dcg